# ⬡ DriveSense AI — Google Colab Runner
**ITU Lahore · BSAI · Software Engineering Final Project**  
Team: Qasim Bin Shahzad (BSAI-24070) · Salman Ejaz Jathol (BSAI-24067)

---
### Instructions
1. Make sure your runtime type is set to **T4 GPU** for hardware acceleration:
   - Go to **Runtime** → **Change runtime type** → select **T4 GPU**.
2. Run all cells in order (**Runtime** → **Run all**).
3. When the Gradio public link appears (an address ending in `.gradio.live`), click it to open the dashboard.
4. Upload your driving video, set your vehicle options, and click **START EVALUATION**.

> **Note:** The temporary public Gradio link is valid for 72 hours. Save it to present your project.

In [ ]:
# ── Cell 1: Install system dependencies ──────────────────────────────────────
!apt-get update -qq
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ ffmpeg installed')

In [ ]:
# ── Cell 2: Install Python packages ──────────────────────────────────────────
!pip install -q ultralytics gradio opencv-python-headless numpy matplotlib
print('✅ Python packages installed')

In [ ]:
%%writefile detector.py
"""
detector.py  –  DriveSense AI  |  Core Detection Engine
=========================================================
Handles:
  • Car / truck detection with own-car masking
  • Real-world distance estimation from bounding-box geometry
  • Lane-line detection via Hough transforms + perspective cues
  • Traffic-light detection + colour classification (R/G/Y)
  • Per-frame event logging for the scorer
"""

import cv2
import numpy as np
from ultralytics import YOLO
from dataclasses import dataclass, field
from typing import Optional
import math

# ─────────────────────────────────────────────
#  CONSTANTS  (tunable via DriverProfile)
# ─────────────────────────────────────────────
COCO_PEDESTRIAN_ID = 0
COCO_BICYCLE_ID    = 1
COCO_CAR_ID        = 2
COCO_MOTORCYCLE_ID = 3
COCO_BUS_ID        = 5
COCO_TRUCK_ID      = 7
COCO_LIGHT_ID      = 9
COCO_STOP_SIGN_ID  = 11

COCO_VEHICLE_IDS   = {2, 5, 7, 3, 1}   # car, bus, truck, motorcycle, bicycle
COCO_CAR_IDS       = {2, 5, 7}

# Representative widths (metres) for distance estimation (Pakistan standard size estimate)
CLASS_WIDTHS = {
    0: 0.55,  # Pedestrian
    1: 0.65,  # Bicycle
    2: 1.75,  # Car (standard)
    3: 0.80,  # Motorcycle
    5: 2.50,  # Bus
    7: 2.50,  # Truck
}

# Approximate focal-length calibration constant (pixels × metres / pixels).
FOCAL_CONST     = 800

# Own-car mask: ignore detections in the bottom-centre strip
OWN_CAR_MASK_FRAC_Y  = 0.82
OWN_CAR_MASK_FRAC_X  = (0.25, 0.75)

# Traffic light & stop sign minimum confidence
LIGHT_MIN_CONF = 0.40
STOP_SIGN_MIN_CONF = 0.40

# ─────────────────────────────────────────────
#  COMPREHENSIVE VEHICLE DATABASE  (PakWheels / OEM specs)
#  Triangle Similarity Distance Calculation:
#    depth = (FOCAL_CONST * real_width_m) / bounding_box_width_px
#  This is far more robust than Y-coordinate methods on hills/curves
# ─────────────────────────────────────────────
VEHICLE_DATABASE = {
    "Suzuki Mehran": {
        "width_m": 1.61,
        "length_m": 3.8,
        "front_m": 1.8,
    },
    "Suzuki Alto": {
        "width_m": 1.62,
        "length_m": 3.86,
        "front_m": 1.7,
    },
    "Suzuki Wagon R": {
        "width_m": 1.68,
        "length_m": 4.17,
        "front_m": 1.8,
    },
    "Toyota Corolla": {
        "width_m": 1.80,
        "length_m": 4.63,
        "front_m": 1.9,
    },
    "Honda Civic": {
        "width_m": 1.80,
        "length_m": 4.63,
        "front_m": 2.0,
    },
    "Honda City": {
        "width_m": 1.68,
        "length_m": 4.42,
        "front_m": 1.9,
    },
    "Toyota Yaris": {
        "width_m": 1.70,
        "length_m": 4.43,
        "front_m": 1.8,
    },
    "Toyota Hilux": {
        "width_m": 1.86,
        "length_m": 5.35,
        "front_m": 2.5,
    },
    "Suzuki Jimny": {
        "width_m": 1.64,
        "length_m": 3.64,
        "front_m": 1.9,
    },
    "Toyota Prius": {
        "width_m": 1.77,
        "length_m": 4.63,
        "front_m": 2.1,
    },
    "Honda N-WGN": {
        "width_m": 1.68,
        "length_m": 3.88,
        "front_m": 1.85,
    },
    "Daihatsu Mira": {
        "width_m": 1.60,
        "length_m": 3.70,
        "front_m": 1.6,
    },
    "Hyundai i10": {
        "width_m": 1.68,
        "length_m": 3.85,
        "front_m": 1.7,
    },
    "KIA Picanto": {
        "width_m": 1.63,
        "length_m": 3.84,
        "front_m": 1.75,
    },
    "Toyota Fortuner": {
        "width_m": 1.86,
        "length_m": 4.83,
        "front_m": 2.2,
    },
    "Chevrolet Bolan": {
        "width_m": 1.68,
        "length_m": 4.25,
        "front_m": 1.95,
    },
    "General Truck": {
        "width_m": 2.50,
        "length_m": 6.0,
        "front_m": 2.8,
    },
    "General Bus": {
        "width_m": 2.50,
        "length_m": 10.0,
        "front_m": 2.5,
    },
    "Custom Vehicle": {
        "width_m": 1.75,
        "length_m": 4.5,
        "front_m": 1.85,
    },
    "Standard Car": {
        "width_m": 1.75,
        "length_m": 4.5,
        "front_m": 1.85,
    },
}

# Backward compatibility: extract front lengths only
VEHICLE_FRONT_LENGTH_M = {k: v["front_m"] for k, v in VEHICLE_DATABASE.items()}

# ─────────────────────────────────────────────
#  DATA STRUCTURES
# ─────────────────────────────────────────────
@dataclass
class DriverProfile:
    """User-supplied vehicle / braking profile."""
    vehicle_name    : str   = "Standard Car"
    braking_100_sec : float = 3.5    # seconds to stop from 100 km/h
    speed_limit_kmh : float = 70.0   # assumed road speed limit

    @property
    def vehicle_spec(self) -> dict:
        """
        Return full vehicle specification (width, length, front offset).
        Falls back to 'Standard Car' if vehicle not in database.
        """
        return VEHICLE_DATABASE.get(self.vehicle_name, VEHICLE_DATABASE["Standard Car"])

    @property
    def vehicle_width_m(self) -> float:
        """Vehicle width in metres (used for triangle similarity distance)."""
        return self.vehicle_spec.get("width_m", 1.75)

    @property
    def vehicle_front_length_m(self) -> float:
        """Camera-to-bumper distance (used when interior is visible)."""
        return self.vehicle_spec.get("front_m", 1.85)

    @property
    def safe_distance_m(self) -> float:
        """
        Safe following distance = reaction_distance + braking_distance.
        Uses the 2-second rule plus the vehicle's own braking distance at speed_limit.
        Formula: d = v × t_reaction + v²/(2a)
        v in m/s, a from braking profile.
        """
        v = self.speed_limit_kmh / 3.6         # m/s
        t_reaction = 1.5                        # s (human reaction)
        # deceleration from braking profile: v0/t  (0→100 reversed)
        v100 = 100 / 3.6
        a = v100 / max(self.braking_100_sec, 0.5)
        d_braking = (v ** 2) / (2 * a)
        d_reaction = v * t_reaction
        # Apply a 1.2× leniency factor as per SRS
        return (d_reaction + d_braking) * 1.2


@dataclass
class FrameEvent:
    """One frame's analysis result."""
    frame_idx       : int
    timestamp_s     : float
    # Collision and Following
    nearest_car_dist_m : Optional[float] = None    # adjusted distance (bumper-to-bumper)
    raw_nearest_dist_m : Optional[float] = None    # raw bounding-box distance
    nearest_car_class   : Optional[int] = None      # COCO class id of nearest obstacle
    interior_visible : bool = False                 # True if steering/meter/dashboard detected
    too_close       : bool = False
    collision_warning : bool = False                # True if TTC < 2.0s
    ttc_s           : Optional[float] = None        # estimated TTC in seconds
    # Lane
    lane_status     : str  = "OK"        # OK | DRIFT | OVER_LINE
    lane_left_line  : Optional[tuple[int, int]] = None  # (x_bottom, x_top)
    lane_right_line : Optional[tuple[int, int]] = None # (x_bottom, x_top)
    # Traffic light
    light_detected  : bool = False
    light_color     : str  = "NONE"      # RED | GREEN | YELLOW | UNKNOWN
    light_violation : bool = False       # car stationary at green too long
    # Stop Sign
    stop_sign_detected : bool = False
    stop_sign_violation : bool = False   # failed to stop when passing stop sign
    # Annotated frame (written by annotate())
    annotated_frame : Optional[np.ndarray] = None


# ─────────────────────────────────────────────
#  DETECTOR CLASS
# ─────────────────────────────────────────────
class DriveSenseDetector:

    def __init__(self, profile: DriverProfile, model_name: str = "yolo11n.pt"):
        print(f"[DriveSense] Loading model: {model_name}")
        self.model   = YOLO(model_name)
        self.profile = profile

        # State for traffic-light green-stationary timer
        self._green_stationary_frames = 0
        self._GREEN_FRAMES_THRESHOLD  = 0  # set per fps in process_video

        # Previous frame for optical flow (crude motion detection)
        self._prev_gray : Optional[np.ndarray] = None

        # Interior visibility detection (rolling average)
        self._interior_visible_count = 0
        self._interior_detection_window = 5  # frames to track

        # Rolling buffer of recent nearest-car detections for TTC calculation
        # Format: list of (timestamp, distance)
        self._distance_history = []
        
        # Stop sign tracking state
        self._stop_sign_visible = False
        self._stop_sign_stopped = False
        self._stop_sign_max_width = 0
        
        # Reference width for distance calibration (defaults to 1080p width)
        self._frame_width = 1920

    # ──────────────────────────────────────────
    #  MAIN PER-FRAME ENTRY POINT
    # ──────────────────────────────────────────
    def analyse_frame(self, frame: np.ndarray, frame_idx: int,
                      fps: float, green_frames_threshold: int) -> FrameEvent:
        h, w = frame.shape[:2]
        self._frame_width = w
        ts   = frame_idx / max(fps, 1)
        event = FrameEvent(frame_idx=frame_idx, timestamp_s=ts)

        # ── 1. YOLO inference ──
        results = self.model(frame, verbose=False, conf=0.35)
        boxes   = results[0].boxes if results else []

        detected_obstacles = [] # list of (xyxy, conf, cls_id)
        light_boxes = []
        stop_sign_boxes = []

        for box in boxes:
            cls_id = int(box.cls[0].item())
            conf   = float(box.conf[0].item())
            xyxy   = box.xyxy[0].cpu().numpy().astype(int)

            if cls_id in COCO_VEHICLE_IDS or cls_id == COCO_PEDESTRIAN_ID:
                if not self._is_own_car(xyxy, h, w):
                    detected_obstacles.append((xyxy, conf, cls_id))
            elif cls_id == COCO_LIGHT_ID and conf >= LIGHT_MIN_CONF:
                light_boxes.append((xyxy, conf))
            elif cls_id == COCO_STOP_SIGN_ID and conf >= STOP_SIGN_MIN_CONF:
                stop_sign_boxes.append((xyxy, conf))

        # ── 2. Interior visibility detection ──
        event.interior_visible = self._detect_interior(frame)
        if event.interior_visible:
            self._interior_visible_count += 1
        else:
            self._interior_visible_count = max(0, self._interior_visible_count - 1)

        # ── 3. Distance estimation ──
        nearest_raw_dist = None
        nearest_dist = None
        nearest_box  = None
        nearest_class = None
        
        for xyxy, conf, cls_id in detected_obstacles:
            raw_dist = self._estimate_distance(xyxy, cls_id)
            if raw_dist is not None and (nearest_raw_dist is None or raw_dist < nearest_raw_dist):
                nearest_raw_dist = raw_dist
                nearest_box  = xyxy
                nearest_class = cls_id

        # Adjust distance based on interior visibility (camera inside car vs dashcam)
        if nearest_raw_dist is not None:
            event.raw_nearest_dist_m = nearest_raw_dist
            event.nearest_car_class = nearest_class
            if self._interior_visible_count > 0:
                # Bumper is closer than camera. Subtract hood offset (physically correct!)
                adjusted_dist = max(0.5, nearest_raw_dist - self.profile.vehicle_front_length_m)
                nearest_dist = adjusted_dist
            else:
                nearest_dist = nearest_raw_dist

        event.nearest_car_dist_m = nearest_dist
        if nearest_dist is not None:
            event.too_close = nearest_dist < self.profile.safe_distance_m

        # ── 4. TTC and Collision Warning (FCW) ──
        if nearest_dist is not None:
            self._distance_history.append((ts, nearest_dist))
            if len(self._distance_history) > 6:
                self._distance_history.pop(0)
                
            # If we have at least 3 points, fit a line to calculate relative velocity (slope)
            if len(self._distance_history) >= 3:
                times = [pt[0] for pt in self._distance_history]
                dists = [pt[1] for pt in self._distance_history]
                slope, _ = np.polyfit(times, dists, 1) # slope is relative velocity (m/s)
                
                # If slope is negative, we are closing in (V_rel is positive closing speed)
                if slope < -0.2:
                    v_closing = -slope
                    ttc = nearest_dist / v_closing
                    event.ttc_s = round(ttc, 2)
                    if ttc < 2.0:
                        event.collision_warning = True
        else:
            self._distance_history.clear()

        # ── 5. Traffic-light colour ──
        if light_boxes:
            event.light_detected = True
            best_xyxy = max(light_boxes, key=lambda x: x[1])[0]
            event.light_color = self._classify_light_color(frame, best_xyxy)

        # ── 6. Green-light stationary check ──
        is_moving = self._is_car_moving(frame)
        if event.light_color == "GREEN" and not is_moving and nearest_dist is None:
            self._green_stationary_frames += 1
            if self._green_stationary_frames >= green_frames_threshold:
                event.light_violation = True
        else:
            self._green_stationary_frames = 0

        # ── 7. Stop Sign check & violations ──
        if stop_sign_boxes:
            event.stop_sign_detected = True
            best_ss_box = max(stop_sign_boxes, key=lambda x: x[1])[0]
            ss_w = best_ss_box[2] - best_ss_box[0]
            self._stop_sign_visible = True
            self._stop_sign_max_width = max(self._stop_sign_max_width, ss_w)
            if not is_moving:
                self._stop_sign_stopped = True
        else:
            # If stop sign was visible but now is gone
            if self._stop_sign_visible:
                # If we passed it closely (width > 25px) and never stopped, trigger violation
                if self._stop_sign_max_width > 25 and not self._stop_sign_stopped:
                    event.stop_sign_violation = True
                # Reset tracking
                self._stop_sign_visible = False
                self._stop_sign_stopped = False
                self._stop_sign_max_width = 0

        # ── 8. Lane detection ──
        event.lane_status, event.lane_left_line, event.lane_right_line = self._check_lane_and_get_lines(frame)

        # ── 9. Annotate frame ──
        event.annotated_frame = self._annotate(
            frame.copy(), event, detected_obstacles, light_boxes, stop_sign_boxes,
            nearest_box, nearest_dist, h, w
        )

        # Store gray for next motion check
        self._prev_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        return event

    # ──────────────────────────────────────────
    #  OWN-CAR MASK
    # ──────────────────────────────────────────
    def _is_own_car(self, xyxy, h, w) -> bool:
        """
        Returns True if this detection is likely the user's own bonnet/dashboard.
        Criteria:
          - Bottom of box is in the very bottom strip  AND
          - Horizontal centre is in the centre band (not a car to the side)
        """
        x1, y1, x2, y2 = xyxy
        box_bottom_frac = y2 / h
        cx_frac         = ((x1 + x2) / 2) / w
        return (box_bottom_frac > OWN_CAR_MASK_FRAC_Y and
                OWN_CAR_MASK_FRAC_X[0] < cx_frac < OWN_CAR_MASK_FRAC_X[1])

    # ──────────────────────────────────────────
    #  DISTANCE ESTIMATION  (Triangle Similarity)
    # ──────────────────────────────────────────
    def _estimate_distance(self, xyxy, cls_id: int) -> Optional[float]:
        """
        Triangle Similarity Distance Estimation
        ═══════════════════════════════════════
        Uses class-specific physical widths to compute distance:
          distance = (FOCAL_CONST * target_width_m) / bounding_box_width_px
        """
        x1, y1, x2, y2 = xyxy
        bbox_width_px = x2 - x1
        
        if bbox_width_px < 8:
            return None
        
        # Get target width based on detected class
        target_width = CLASS_WIDTHS.get(cls_id, 1.75)
        
        # Scale focal constant relative to 1080p width (1920px) to handle rescaled/low-res streams
        adjusted_focal = FOCAL_CONST * (self._frame_width / 1920.0)
        
        distance_m = (adjusted_focal * target_width) / bbox_width_px
        return round(distance_m, 1)

    # ──────────────────────────────────────────
    #  TRAFFIC LIGHT COLOUR
    # ──────────────────────────────────────────
    def _classify_light_color(self, frame: np.ndarray, xyxy) -> str:
        """
        Classify traffic light colour by dominant HSV hue in the crop.
        Strategy: split crop into top / middle / bottom thirds and find
        the brightest (most saturated) third, then check its hue.
        """
        x1, y1, x2, y2 = xyxy
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
        crop = frame[y1:y2, x1:x2]
        if crop.size == 0:
            return "UNKNOWN"

        hsv  = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        h_c  = crop.shape[0]

        thirds = [
            hsv[:h_c//3],
            hsv[h_c//3 : 2*h_c//3],
            hsv[2*h_c//3:]
        ]
        names    = ["RED", "YELLOW", "GREEN"]
        # Rough hue ranges: Red ≈ 0-10 or 160-180, Yellow ≈ 20-35, Green ≈ 40-85
        hue_ranges = [
            [(0, 15), (160, 180)],
            [(20, 38)],
            [(38, 90)]
        ]

        scores = []
        for third, ranges in zip(thirds, hue_ranges):
            mask = np.zeros(third.shape[:2], dtype=np.uint8)
            for lo, hi in ranges:
                mask |= cv2.inRange(third, (lo, 60, 60), (hi, 255, 255))
            scores.append(int(mask.sum()))

        best = int(np.argmax(scores))
        # Require a minimum pixel count to avoid noise
        if scores[best] < 50:
            return "UNKNOWN"
        return names[best]

    # ──────────────────────────────────────────
    #  MOTION DETECTION  (is the user's car moving?)
    # ──────────────────────────────────────────
    def _is_car_moving(self, frame: np.ndarray) -> bool:
        """
        Crude optical-flow proxy: compare centre ROI of current vs previous frame.
        Returns True if significant motion is detected.
        """
        if self._prev_gray is None:
            return True  # assume moving on first frame
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        # Use centre strip (avoids edge-of-frame tree/building parallax)
        roi_prev = self._prev_gray[h//4 : 3*h//4, w//4 : 3*w//4]
        roi_curr = gray[h//4 : 3*h//4, w//4 : 3*w//4]
        diff     = cv2.absdiff(roi_prev, roi_curr)
        motion   = float(diff.mean())
        return motion > 1.5   # pixels threshold — tunable

    # ──────────────────────────────────────────
    #  INTERIOR DETECTION  (steering wheel / dashboard)
    # ──────────────────────────────────────────
    def _detect_interior(self, frame: np.ndarray) -> bool:
        """
        Detect if steering wheel or dashboard is visible (wearable camera).
        Strategy:
          1. Check bottom-left and bottom-right corners for dark circular structures (steering wheel)
          2. Check bottom-centre for high-texture areas (dashboard, steering)
          3. Analyse color distribution: interiors have more blacks/dark grays
        Returns True if interior indicators are found.
        """
        h, w = frame.shape[:2]
        
        # Sample three regions: left, centre, right of bottom strip
        bottom_strip_h = int(h * 0.35)   # bottom 35% of frame
        if bottom_strip_h < 10:
            return False
        
        bottom_region = frame[-bottom_strip_h:, :]
        gray = cv2.cvtColor(bottom_region, cv2.COLOR_BGR2GRAY)
        
        # Detect high-texture areas (steering wheel, dashboard controls)
        # Use Laplacian for edge/texture detection
        laplacian = cv2.Laplacian(gray, cv2.CV_64F)
        texture = np.abs(laplacian).mean()
        
        # Dark pixels indicate interior (steering wheel is typically dark)
        dark_pct = (gray < 80).sum() / gray.size
        
        # Steering wheels are often symmetric circular patterns → check for circular edges
        edges = cv2.Canny(gray, 50, 150)
        circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, dp=1, minDist=30,
                                    param1=50, param2=30, minRadius=15, maxRadius=80)
        has_circles = circles is not None and len(circles[0]) > 0
        
        # Thresholds for interior detection
        # High texture + dark pixels + circular patterns = likely interior
        interior_score = 0
        if texture > 15:  # significant texture
            interior_score += 1
        if dark_pct > 0.25:  # >25% dark pixels
            interior_score += 1
        if has_circles:  # circular steering wheel detected
            interior_score += 2
        
        return interior_score >= 2

    # ──────────────────────────────────────────
    #  LANE DETECTION
    # ──────────────────────────────────────────
    # ──────────────────────────────────────────
    #  LANE DETECTION (Polynomial Fitting)
    # ──────────────────────────────────────────
    def _check_lane_and_get_lines(self, frame: np.ndarray) -> tuple[str, Optional[tuple[int, int]], Optional[tuple[int, int]]]:
        h, w = frame.shape[:2]

        # ROI: trapezoid covering the lower road area
        roi_vertices = np.array([[
            (int(w * 0.0), h),
            (int(w * 0.45), int(h * 0.55)),
            (int(w * 0.55), int(h * 0.55)),
            (int(w * 1.0), h),
        ]], dtype=np.int32)

        gray    = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur    = cv2.GaussianBlur(gray, (7, 7), 0)
        edges   = cv2.Canny(blur, 40, 120)

        # Mask to ROI
        mask    = np.zeros_like(edges)
        cv2.fillPoly(mask, roi_vertices, 255)
        masked  = cv2.bitwise_and(edges, mask)

        lines   = cv2.HoughLinesP(
            masked, rho=1, theta=np.pi/180,
            threshold=40, minLineLength=50, maxLineGap=80
        )

        if lines is None:
            return "OK", None, None

        left_pts, right_pts = [], []

        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 == x1:
                continue
            slope = (y2 - y1) / (x2 - x1)
            # Steep enough to be a lane line
            if abs(slope) < 0.3 or abs(slope) > 5.0:
                continue

            cx = (x1 + x2) // 2
            if slope < 0 and cx < w * 0.55:   # left lane line
                left_pts.append((x1, y1))
                left_pts.append((x2, y2))
            elif slope > 0 and cx > w * 0.45:  # right lane line
                right_pts.append((x1, y1))
                right_pts.append((x2, y2))

        left_line = None
        right_line = None
        y_bottom = h
        y_top = int(h * 0.6)

        # Fit left line: x = a*y + b
        if len(left_pts) >= 2:
            ys = [p[1] for p in left_pts]
            xs = [p[0] for p in left_pts]
            try:
                a, b = np.polyfit(ys, xs, 1)
                x_bottom = int(a * y_bottom + b)
                x_top = int(a * y_top + b)
                if -w < x_bottom < 2*w:
                    left_line = (x_bottom, x_top)
            except np.linalg.LinAlgError:
                pass

        # Fit right line: x = a*y + b
        if len(right_pts) >= 2:
            ys = [p[1] for p in right_pts]
            xs = [p[0] for p in right_pts]
            try:
                a, b = np.polyfit(ys, xs, 1)
                x_bottom = int(a * y_bottom + b)
                x_top = int(a * y_top + b)
                if -w < x_bottom < 2*w:
                    right_line = (x_bottom, x_top)
            except np.linalg.LinAlgError:
                pass

        centre  = w // 2
        
        # Over-line: if wheels are close/crossing the lane markers
        OVER_THRESH = int(w * 0.08)   # 8% of width
        if left_line is not None and abs(left_line[0] - centre) < OVER_THRESH:
            return "OVER_LINE", left_line, right_line
        if right_line is not None and abs(right_line[0] - centre) < OVER_THRESH:
            return "OVER_LINE", left_line, right_line

        # Drift: only one line visible
        if left_line is not None and right_line is None:
            return "DRIFT", left_line, right_line
        if right_line is not None and left_line is None:
            return "DRIFT", left_line, right_line

        if not left_line and not right_line:
            return "OK", None, None

        return "OK", left_line, right_line

    # ──────────────────────────────────────────
    #  LANE OVERLAY DRAWING
    # ──────────────────────────────────────────
    def _draw_lane_overlay(self, frame, event: FrameEvent, h, w) -> np.ndarray:
        status = event.lane_status
        left = event.lane_left_line
        right = event.lane_right_line
        
        # BGR Colors
        color_map = {
            "OK": (80, 220, 80),       # Soft green
            "DRIFT": (0, 165, 255),    # Soft orange
            "OVER_LINE": (0, 0, 220)   # Soft red
        }
        color = color_map.get(status, (80, 220, 80))
        
        overlay = frame.copy()
        y_bottom = h
        y_top = int(h * 0.6)
        
        # Draw transparent lane polygon if both lines are detected
        if left is not None and right is not None:
            pts = np.array([
                [left[0], y_bottom],
                [left[1], y_top],
                [right[1], y_top],
                [right[0], y_bottom]
            ], dtype=np.int32)
            cv2.fillPoly(overlay, [pts], color)
            cv2.addWeighted(overlay, 0.22, frame, 0.78, 0, frame)
            
            # Draw line boundaries
            cv2.line(frame, (left[0], y_bottom), (left[1], y_top), color, 3, cv2.LINE_AA)
            cv2.line(frame, (right[0], y_bottom), (right[1], y_top), color, 3, cv2.LINE_AA)
        elif left is not None:
            cv2.line(frame, (left[0], y_bottom), (left[1], y_top), color, 3, cv2.LINE_AA)
        elif right is not None:
            cv2.line(frame, (right[0], y_bottom), (right[1], y_top), color, 3, cv2.LINE_AA)
            
        return frame

    # ──────────────────────────────────────────
    #  ANNOTATION
    # ──────────────────────────────────────────
    def _annotate(self, frame, event: FrameEvent,
                  detected_obstacles, light_boxes, stop_sign_boxes,
                  nearest_box, nearest_dist, h, w) -> np.ndarray:

        # ── 1. Draw lane overlay first ──
        frame = self._draw_lane_overlay(frame, event, h, w)

        # ── 2. Draw detected obstacles ──
        for xyxy, conf, cls_id in detected_obstacles:
            dist = self._estimate_distance(xyxy, cls_id)
            is_nearest = nearest_box is not None and np.array_equal(xyxy, nearest_box)
            
            # Determine color and labels based on class
            if cls_id == COCO_PEDESTRIAN_ID:
                color = (255, 120, 0) # Cyan/Blue-ish in BGR
                label = f"Pedestrian {dist:.1f}m" if dist else "pedestrian"
            elif cls_id == COCO_MOTORCYCLE_ID or cls_id == COCO_BICYCLE_ID:
                color = (255, 200, 0) # Yellow/Blue
                label = f"Cycle {dist:.1f}m" if dist else "cycle"
            else: # Car, bus, truck
                color = (0, 0, 220) if (is_nearest and event.too_close) else (50, 205, 50)
                label = f"Vehicle {dist:.1f}m" if dist else "vehicle"
                if is_nearest and event.too_close:
                    label += " [TOO CLOSE]"
                if is_nearest and event.collision_warning:
                    label += " ⚠ FCW ⚠"
                    color = (0, 0, 255) # Bright Red

            # Draw bounding box
            thickness = 3 if (is_nearest and (event.too_close or event.collision_warning)) else 2
            cv2.rectangle(frame, (xyxy[0], xyxy[1]), (xyxy[2], xyxy[3]), color, thickness)
            self._put_label(frame, label, (xyxy[0], xyxy[1] - 8), color)

        # ── 3. Draw traffic light boxes ──
        for xyxy, conf in light_boxes:
            color_map = {"RED": (0, 0, 255), "GREEN": (0, 220, 0),
                         "YELLOW": (0, 210, 255), "UNKNOWN": (200, 200, 200)}
            c = color_map.get(event.light_color, (200, 200, 200))
            cv2.rectangle(frame, (xyxy[0], xyxy[1]), (xyxy[2], xyxy[3]), c, 2)
            self._put_label(frame, f"LIGHT:{event.light_color}", (xyxy[0], xyxy[1] - 8), c)

        # ── 4. Draw stop sign boxes ──
        for xyxy, conf in stop_sign_boxes:
            c = (0, 0, 255) # Red box
            cv2.rectangle(frame, (xyxy[0], xyxy[1]), (xyxy[2], xyxy[3]), c, 3)
            label = "STOP SIGN"
            if event.stop_sign_violation:
                label += " (VIOLATION)"
            self._put_label(frame, label, (xyxy[0], xyxy[1] - 8), c)

        # ── 5. HUD overlay (semi-transparent panel at the top) ──
        panel_h = 95
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, panel_h), (20, 16, 16), -1)
        cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, frame)

        # Distance & TTC Info
        if nearest_dist is not None:
            safe_d = self.profile.safe_distance_m
            dist_col = (0, 80, 255) if event.too_close else (60, 220, 60)
            if event.collision_warning:
                dist_col = (0, 0, 255) # Red for critical
                
            dist_txt = f"GAP: {nearest_dist:.1f}m  |  SAFE: {safe_d:.1f}m"
            if event.ttc_s is not None:
                dist_txt += f"  |  TTC: {event.ttc_s:.1f}s"
            if event.interior_visible:
                dist_txt += " [CABIN]"
            cv2.putText(frame, dist_txt, (12, 28),
                        cv2.FONT_HERSHEY_DUPLEX, 0.65, dist_col, 1, cv2.LINE_AA)
        else:
            cv2.putText(frame, "GAP: --  |  SAFE: --", (12, 28),
                        cv2.FONT_HERSHEY_DUPLEX, 0.65, (180, 180, 180), 1, cv2.LINE_AA)

        # Lane status
        lane_col = {"OK": (60, 220, 60),
                    "DRIFT": (0, 180, 255),
                    "OVER_LINE": (0, 60, 255)}.get(event.lane_status, (180, 180, 180))
        cv2.putText(frame, f"LANE STATUS: {event.lane_status}", (12, 58),
                    cv2.FONT_HERSHEY_DUPLEX, 0.65, lane_col, 1, cv2.LINE_AA)

        # Light & Stop Sign Status
        sig_col = (180, 180, 180)
        sig_txt = "SYSTEMS ONLINE"
        
        if event.light_detected:
            sig_txt = f"TRAFFIC LIGHT: {event.light_color}"
            sig_col = (0, 220, 0) if event.light_color == "GREEN" else (0, 0, 255)
            if event.light_violation:
                sig_txt += " (GREEN LIGHT DELAY VIOLATION)"
                sig_col = (0, 0, 255)
        elif event.stop_sign_detected:
            sig_txt = "STOP SIGN DETECTED"
            sig_col = (0, 120, 255)
            if event.stop_sign_violation:
                sig_txt += " (STOP SIGN ROLL VIOLATION)"
                sig_col = (0, 0, 255)
                
        cv2.putText(frame, sig_txt, (12, 88),
                    cv2.FONT_HERSHEY_DUPLEX, 0.65, sig_col, 1, cv2.LINE_AA)

        # Simulated Speedometer
        is_moving = self._is_car_moving(frame)
        if is_moving:
            simulated_speed = int(self.profile.speed_limit_kmh * 0.9 + (event.frame_idx % 12 - 6) * 0.5)
        else:
            simulated_speed = 0
            
        speed_txt = f"{simulated_speed} KM/H"
        cv2.putText(frame, speed_txt, (w - 150, 58),
                    cv2.FONT_HERSHEY_DUPLEX, 0.8, (0, 220, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, f"LIMIT: {int(self.profile.speed_limit_kmh)}", (w - 150, 80),
                    cv2.FONT_HERSHEY_DUPLEX, 0.45, (120, 120, 120), 1, cv2.LINE_AA)

        # Flashing HUD Warnings
        warning_active = event.collision_warning or event.lane_status in {"DRIFT", "OVER_LINE"} or event.stop_sign_violation or event.light_violation
        if warning_active and (event.frame_idx // 3) % 2 == 0:
            if event.collision_warning:
                cv2.rectangle(frame, (w//2 - 180, h - 80), (w//2 + 180, h - 30), (0, 0, 220), -1)
                cv2.putText(frame, "COLLISION WARNING (TTC < 2s)", (w//2 - 160, h - 48),
                            cv2.FONT_HERSHEY_DUPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
            elif event.lane_status == "OVER_LINE":
                cv2.rectangle(frame, (w//2 - 180, h - 80), (w//2 + 180, h - 30), (0, 69, 255), -1)
                cv2.putText(frame, "LANE DEPARTURE WARNING", (w//2 - 145, h - 48),
                            cv2.FONT_HERSHEY_DUPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)

        # Timestamp
        ts_txt = f"{event.timestamp_s:.1f}s"
        cv2.putText(frame, ts_txt, (w - 100, 28),
                    cv2.FONT_HERSHEY_DUPLEX, 0.6, (160, 160, 160), 1, cv2.LINE_AA)

        return frame

    # ──────────────────────────────────────────
    #  HELPERS
    # ──────────────────────────────────────────
    @staticmethod
    def _put_label(frame, text, pos, color, scale=0.5, thick=1):
        """Draw a label with a dark background for readability."""
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, thick)
        x, y = pos
        cv2.rectangle(frame, (x - 2, y - th - 4), (x + tw + 2, y + 2),
                      (10, 10, 10), -1)
        cv2.putText(frame, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                    scale, color, thick, cv2.LINE_AA)


In [ ]:
%%writefile scorer.py
"""
scorer.py  –  DriveSense AI  |  Safety Scoring & Report Engine
===============================================================
Converts a list of FrameEvent objects into:
  • A numeric safety score  (0.0 – 10.0)
  • A structured report dict with per-category breakdowns
  • A human-readable markdown report string
  • Personalised improvement suggestions
  • A chronological timeline of driving incidents
"""

from __future__ import annotations
from typing import List, Dict, Any
from dataclasses import dataclass, field
from detector import FrameEvent, DriverProfile


# ─────────────────────────────────────────────
#  SCORING WEIGHTS  (must sum to 1.0)
# ─────────────────────────────────────────────
WEIGHTS = {
    "following_distance" : 0.40,   # following gap & forward collision risk
    "lane_discipline"    : 0.35,   # lane drift & over-line infractions
    "intersection_safety": 0.25,   # traffic signals & stop signs
}

# How much the score drops per infraction (out of 10)
PENALTIES = {
    "too_close_per_frame"    : 0.15,   # tailgating
    "lane_drift_per_event"   : 0.8,    # lane drift
    "lane_over_per_event"    : 1.2,    # wheels over line
    "light_violation"        : 2.5,    # green-light stationary delay
    "stop_sign_violation"    : 2.0,    # rolling through stop sign
    "collision_warning_event": 1.5,    # critical TTC event
}


# ─────────────────────────────────────────────
#  REPORT DATACLASS
# ─────────────────────────────────────────────
@dataclass
class SafetyReport:
    overall_score       : float
    distance_score      : float
    lane_score          : float
    light_score         : float  # Intersection safety (retained name for UI compatibility)

    total_frames        : int
    analysed_frames     : int
    duration_s          : float

    too_close_count     : int
    drift_events        : int
    over_line_events    : int
    light_violations    : int
    lights_seen         : int

    vehicle_name        : str
    suggestions         : List[str]
    markdown            : str
    
    stop_sign_violations: int = 0
    stop_signs_seen     : int = 0
    collision_warnings  : int = 0
    timeline            : List[Dict[str, Any]] = field(default_factory=list)


# ─────────────────────────────────────────────
#  SCORER
# ─────────────────────────────────────────────
class SafetyScorer:

    def __init__(self, profile: DriverProfile):
        self.profile = profile

    def score(self, events: List[FrameEvent]) -> SafetyReport:
        if not events:
            return self._empty_report()

        n = len(events)
        duration_s = events[-1].timestamp_s

        # ── 1. Following Distance & Collision Warnings ──────────────────
        too_close_frames = sum(1 for e in events if e.too_close)
        car_visible = sum(1 for e in events if e.nearest_car_dist_m is not None)
        
        if car_visible > 0:
            close_frac  = too_close_frames / car_visible
            dist_penalty = min(8.0, close_frac * 15.0)  # max 8 points penalty for tailgating
        else:
            dist_penalty = 0.0
            
        collision_events = self._count_collision_events(events)
        collision_penalty = min(5.0, collision_events * PENALTIES["collision_warning_event"])
        
        distance_score = max(0.0, 10.0 - dist_penalty - collision_penalty)

        # ── 2. Lane Discipline ─────────────────────
        drift_events, over_line_events = self._count_lane_events(events)
        lane_penalty = (drift_events    * PENALTIES["lane_drift_per_event"] +
                        over_line_events * PENALTIES["lane_over_per_event"])
        lane_score   = max(0.0, 10.0 - lane_penalty)

        # ── 3. Intersection Safety (Traffic Lights & Stop Signs) ──────
        light_violations = sum(1 for e in events if e.light_violation)
        lights_seen      = sum(1 for e in events if e.light_detected)
        
        stop_sign_violations = sum(1 for e in events if e.stop_sign_violation)
        stop_signs_seen      = sum(1 for e in events if e.stop_sign_detected)
        
        intersection_penalty = (light_violations * PENALTIES["light_violation"] +
                                stop_sign_violations * PENALTIES["stop_sign_violation"])
        intersection_score   = max(0.0, 10.0 - intersection_penalty)

        # ── 4. Weighted Overall ────────────────────
        overall = (
            WEIGHTS["following_distance"] * distance_score +
            WEIGHTS["lane_discipline"]    * lane_score     +
            WEIGHTS["intersection_safety"] * intersection_score
        )
        overall = round(min(10.0, max(0.0, overall)), 1)

        # ── 5. Build chronological incident timeline ──
        timeline = self._build_timeline(events)

        # ── 6. Suggestions ────────────────────────
        suggestions = self._build_suggestions(
            distance_score, lane_score, intersection_score,
            too_close_frames, drift_events, over_line_events,
            light_violations, stop_sign_violations, collision_events
        )

        # ── 7. Markdown report ────────────────────
        md = self._build_markdown(
            overall, distance_score, lane_score, intersection_score,
            n, duration_s, too_close_frames, drift_events,
            over_line_events, light_violations, stop_sign_violations,
            collision_events, lights_seen, stop_signs_seen, suggestions, timeline
        )

        return SafetyReport(
            overall_score    = overall,
            distance_score   = round(distance_score, 1),
            lane_score       = round(lane_score, 1),
            light_score      = round(intersection_score, 1), # mapped to light_score for UI
            total_frames     = n,
            analysed_frames  = n,
            duration_s       = round(duration_s, 1),
            too_close_count  = too_close_frames,
            drift_events     = drift_events,
            over_line_events = over_line_events,
            light_violations = light_violations,
            lights_seen      = lights_seen,
            stop_sign_violations = stop_sign_violations,
            stop_signs_seen  = stop_signs_seen,
            collision_warnings = collision_events,
            timeline         = timeline,
            vehicle_name     = self.profile.vehicle_name,
            suggestions      = suggestions,
            markdown         = md,
        )

    # ──────────────────────────────────────────
    #  COUNT LANE EVENTS
    # ──────────────────────────────────────────
    @staticmethod
    def _count_lane_events(events: List[FrameEvent]):
        drift_events = over_line_events = 0
        prev = "OK"
        for e in events:
            s = e.lane_status
            if s == "DRIFT"     and prev != "DRIFT":
                drift_events += 1
            elif s == "OVER_LINE" and prev != "OVER_LINE":
                over_line_events += 1
            prev = s
        return drift_events, over_line_events

    # ──────────────────────────────────────────
    #  COUNT COLLISION WARNINGS
    # ──────────────────────────────────────────
    @staticmethod
    def _count_collision_events(events: List[FrameEvent]) -> int:
        count = 0
        prev = False
        for e in events:
            curr = e.collision_warning
            if curr and not prev:
                count += 1
            prev = curr
        return count

    # ──────────────────────────────────────────
    #  BUILD CHRONOLOGICAL TIMELINE
    # ──────────────────────────────────────────
    def _build_timeline(self, events: List[FrameEvent]) -> List[Dict[str, Any]]:
        timeline = []
        
        in_tailgate = False
        tailgate_start_ts = 0.0
        
        in_fcw = False
        fcw_start_ts = 0.0
        
        in_drift = False
        drift_start_ts = 0.0
        
        in_over = False
        over_start_ts = 0.0
        
        for e in events:
            ts = e.timestamp_s
            time_str = f"{int(ts // 60):02d}:{int(ts % 60):02d}"
            
            # Tailgating
            if e.too_close:
                if not in_tailgate:
                    in_tailgate = True
                    tailgate_start_ts = ts
            else:
                if in_tailgate:
                    duration = ts - tailgate_start_ts
                    if duration > 1.0:
                        timeline.append({
                            "timestamp": tailgate_start_ts,
                            "time_str": f"{int(tailgate_start_ts // 60):02d}:{int(tailgate_start_ts % 60):02d}",
                            "type": "⚠️ Tailgating",
                            "severity": "Warning",
                            "message": f"Tailgating vehicle (gap fell to {e.nearest_car_dist_m or 0:.1f}m, safe buffer is {self.profile.safe_distance_m:.1f}m)."
                        })
                    in_tailgate = False
                    
            # Collision Warning (FCW)
            if e.collision_warning:
                if not in_fcw:
                    in_fcw = True
                    fcw_start_ts = ts
            else:
                if in_fcw:
                    timeline.append({
                        "timestamp": fcw_start_ts,
                        "time_str": f"{int(fcw_start_ts // 60):02d}:{int(fcw_start_ts % 60):02d}",
                        "type": "🚨 Collision Alert",
                        "severity": "Danger",
                        "message": f"Forward Collision Warning! Braking time-to-collision (TTC) dropped to {e.ttc_s or 0.0:.1f}s."
                    })
                    in_fcw = False
                    
            # Lane Drift
            if e.lane_status == "DRIFT":
                if not in_drift:
                    in_drift = True
                    drift_start_ts = ts
            else:
                if in_drift:
                    timeline.append({
                        "timestamp": drift_start_ts,
                        "time_str": f"{int(drift_start_ts // 60):02d}:{int(drift_start_ts % 60):02d}",
                        "type": "🛣️ Lane Drift",
                        "severity": "Warning",
                        "message": "Vehicle drifted from the center path without lane markings on both sides."
                    })
                    in_drift = False
                    
            # Over Lane
            if e.lane_status == "OVER_LINE":
                if not in_over:
                    in_over = True
                    over_start_ts = ts
            else:
                if in_over:
                    timeline.append({
                        "timestamp": over_start_ts,
                        "time_str": f"{int(over_start_ts // 60):02d}:{int(over_start_ts % 60):02d}",
                        "type": "⛔ Lane straddle",
                        "severity": "Danger",
                        "message": "Car tyres crossed the lane boundaries. Active lane departure warning."
                    })
                    in_over = False
                    
            # Traffic Light Violation
            if e.light_violation:
                last_violations = [t for t in timeline if t["type"] == "🚦 Traffic Signal Delay"]
                if not last_violations or (ts - last_violations[-1]["timestamp"] > 5.0):
                    timeline.append({
                        "timestamp": ts,
                        "time_str": time_str,
                        "type": "🚦 Traffic Signal Delay",
                        "severity": "Danger",
                        "message": "Failed to proceed at a green traffic light (stationary for > 2 seconds)."
                    })
                    
            # Stop Sign Violation
            if e.stop_sign_violation:
                last_violations = [t for t in timeline if t["type"] == "🛑 Stop Sign Roll"]
                if not last_violations or (ts - last_violations[-1]["timestamp"] > 5.0):
                    timeline.append({
                        "timestamp": ts,
                        "time_str": time_str,
                        "type": "🛑 Stop Sign Roll",
                        "severity": "Danger",
                        "message": "Failed to stop at a stop sign (rolled past without stopping)."
                    })
                    
        # Flush remaining active states
        if in_tailgate:
            timeline.append({
                "timestamp": tailgate_start_ts,
                "time_str": f"{int(tailgate_start_ts // 60):02d}:{int(tailgate_start_ts % 60):02d}",
                "type": "⚠️ Tailgating",
                "severity": "Warning",
                "message": "Tailgating vehicle ahead at the end of the video."
            })
        if in_fcw:
            timeline.append({
                "timestamp": fcw_start_ts,
                "time_str": f"{int(fcw_start_ts // 60):02d}:{int(fcw_start_ts % 60):02d}",
                "type": "🚨 Collision Alert",
                "severity": "Danger",
                "message": "Active collision risk at the end of the video."
            })
        if in_drift:
            timeline.append({
                "timestamp": drift_start_ts,
                "time_str": f"{int(drift_start_ts // 60):02d}:{int(drift_start_ts % 60):02d}",
                "type": "🛣️ Lane Drift",
                "severity": "Warning",
                "message": "Vehicle drifting at the end of the video."
            })
        if in_over:
            timeline.append({
                "timestamp": over_start_ts,
                "time_str": f"{int(over_start_ts // 60):02d}:{int(over_start_ts % 60):02d}",
                "type": "⛔ Lane straddle",
                "severity": "Danger",
                "message": "Vehicle straddling lane line at the end of the video."
            })
            
        timeline.sort(key=lambda x: x["timestamp"])
        return timeline

    # ──────────────────────────────────────────
    #  SUGGESTIONS
    # ──────────────────────────────────────────
    def _build_suggestions(
        self, dist_sc, lane_sc, inter_sc,
        too_close, drifts, over_lines, light_viols, stop_viols, collision_events
    ) -> List[str]:
        tips = []

        if dist_sc < 7.0:
            safe_d = self.profile.safe_distance_m
            tips.append(
                f"⚠️ **Following Gap**: Keep a larger safety buffer. At {self.profile.speed_limit_kmh:.0f} km/h, "
                f"your {self.profile.vehicle_name} needs **{safe_d:.0f} m** to stop in an emergency. "
                f"Apply the '3-Second Rule' to estimate gaps on the highway."
            )
        if collision_events > 0:
            tips.append(
                f"🚨 **Collision Risk**: The system detected {collision_events} critical closing rate events. "
                f"Look further down the road to anticipate decelerations early rather than reacting at the last second."
            )
        elif dist_sc >= 9.0:
            tips.append("✅ **Following Distance**: Good. Kept safe margins from obstacles.")

        if over_lines > 0:
            tips.append(
                f"🛣️ **Lane Markings**: You straddled lane boundaries {over_lines} times. "
                f"Make sure to use indicators before changing lanes, and center yourself between lane markings."
            )
        if drifts > 0:
            tips.append(
                f"⚠️ **Steering Control**: System detected {drifts} drift events. Keep two hands on the wheel "
                f"and avoid driving distractions (mobile phones, dashboard screens)."
            )
        if lane_sc >= 9.5:
            tips.append("✅ **Lane Keeping**: Excellent lane discipline and stability.")

        if light_viols > 0:
            tips.append(
                f"🚦 **Signal Awareness**: Delayed reaction at green light ({light_viols} times). "
                f"Pay active attention at intersections to keep traffic flowing safely."
            )
        if stop_viols > 0:
            tips.append(
                f"🛑 **Stop Signs**: Rolled through {stop_viols} stop sign junctions. "
                f"You must come to a complete stop (0 km/h) at a stop line, scan the intersection, and then proceed."
            )
        if inter_sc == 10.0:
            tips.append("✅ **Intersection Safety**: Perfect response to traffic signals and stop lines.")

        return tips

    # ──────────────────────────────────────────
    #  MARKDOWN REPORT GENERATOR
    # ──────────────────────────────────────────
    def _build_markdown(
        self, overall, dist_sc, lane_sc, inter_sc,
        frames, duration, too_close, drifts, over_lines,
        light_viols, stop_viols, collision_events, lights_seen, stop_signs_seen,
        suggestions, timeline
    ) -> str:

        def stars(s):
            filled = round(s / 2)
            return "★" * filled + "☆" * (5 - filled)

        grade_map = [(9, "A+"), (8, "A"), (7, "B"), (6, "C"), (5, "D"), (0, "F")]
        grade = next(g for threshold, g in grade_map if overall >= threshold)

        lines = [
            f"# 🚗 DriveSense AI — Safety Scorecard",
            f"",
            f"**Vehicle Profile:** {self.profile.vehicle_name}  |  "
            f"**Clip Duration:** {duration:.1f}s  |  "
            f"**Analysed Frames:** {frames}",
            f"",
            f"---",
            f"",
            f"## Overall Score:  **{overall} / 10**   ({grade})",
            f"",
            f"| Driving Category | Score | Star Rating |",
            f"| :--- | :--- | :--- |",
            f"| 🚘 Following Distance & TTC | {dist_sc:.1f} / 10 | {stars(dist_sc)} |",
            f"| 🛣️ Lane Keeping & Discipline | {lane_sc:.1f} / 10 | {stars(lane_sc)} |",
            f"| 🚦 Intersection Safety | {inter_sc:.1f} / 10 | {stars(inter_sc)} |",
            f"",
            f"---",
            f"",
            f"## Incidents & Telemetry Summary",
            f"",
            f"- **Unsafe Following Frames:** {too_close} frames",
            f"- **Collision Warning Events (TTC < 2s):** {collision_events} warnings",
            f"- **Lane Drifts:** {drifts} times",
            f"- **Lane Marker Straddling:** {over_lines} times",
            f"- **Green Light Delay Incidents:** {light_viols} times",
            f"- **Stop Sign Violations:** {stop_viols} times",
            f"",
            f"---",
            f"",
            f"## 📋 Timeline of Driving Incidents",
            f""
        ]
        
        if timeline:
            lines.append("| Time | Incident Type | Description | Severity |")
            lines.append("| :--- | :--- | :--- | :--- |")
            for item in timeline:
                sev_badge = f"<span style='color:#ff5555;font-weight:bold;'>{item['severity']}</span>" if item['severity'] == "Danger" else f"<span style='color:#ffaa00;font-weight:bold;'>{item['severity']}</span>"
                lines.append(f"| **{item['time_str']}** | {item['type']} | {item['message']} | {sev_badge} |")
        else:
            lines.append("🏆 *No critical incidents detected! Perfect driving log.*")
            
        lines.extend([
            f"",
            f"---",
            f"",
            f"## 💡 Personalised Feedback & Suggestions",
            f"",
        ])
        for tip in suggestions:
            lines.append(f"- {tip}")
            
        lines.extend([
            f"",
            f"---",
            f"",
            f"*Generated by DriveSense AI (Semester Project Submission).*",
        ])
        return "\n".join(lines)

    # ──────────────────────────────────────────
    #  EMPTY FALLBACK
    # ──────────────────────────────────────────
    def _empty_report(self) -> SafetyReport:
        return SafetyReport(
            overall_score=0.0, distance_score=0.0, lane_score=0.0,
            light_score=0.0, total_frames=0, analysed_frames=0, duration_s=0.0,
            too_close_count=0, drift_events=0, over_line_events=0,
            light_violations=0, lights_seen=0, stop_sign_violations=0, stop_signs_seen=0,
            collision_warnings=0, timeline=[],
            vehicle_name=self.profile.vehicle_name,
            suggestions=["No frames were analysed. Please upload a valid video."],
            markdown="# Error\n\nNo frames were analysed.",
        )


In [ ]:
%%writefile user_profile.py
"""
user_profile.py  –  DriveSense AI  |  User Vehicle Profile & History Management
=================================================================================
Manages user vehicle preferences and driving run history:
  • Store multiple vehicles associated with user
  • Track usage frequency (primary vehicle detection)
  • Store historical safety runs for 1v1 battle mode and analytics
  • Persist preferences to local JSON storage
"""

import json
import os
import time
from typing import Dict, List, Optional
from pathlib import Path


# Local storage path for user profiles
PROFILES_DIR = Path(os.path.expanduser("~/.drivesense"))
USER_PROFILE_FILE = PROFILES_DIR / "user_vehicles.json"


def ensure_storage():
    """Ensure the storage directory exists."""
    PROFILES_DIR.mkdir(parents=True, exist_ok=True)


class UserProfile:
    """Manages a user's vehicle collection, preferences, and session history."""

    def __init__(self):
        """Initialize user profile, loading from disk if it exists."""
        ensure_storage()
        self.vehicles: Dict[str, dict] = {}  # vehicle_name -> {braking_time, usage_count}
        self.primary_vehicle: Optional[str] = None
        self.history: List[dict] = []        # list of safety analysis runs
        self._load()

    def _load(self):
        """Load profile and history from JSON file."""
        if USER_PROFILE_FILE.exists():
            try:
                data = json.load(open(USER_PROFILE_FILE, "r"))
                self.vehicles = data.get("vehicles", {})
                self.primary_vehicle = data.get("primary_vehicle")
                self.history = data.get("history", [])
            except (json.JSONDecodeError, IOError):
                self.vehicles = {}
                self.primary_vehicle = None
                self.history = []

    def _save(self):
        """Persist profile and history to JSON file."""
        ensure_storage()
        data = {
            "vehicles": self.vehicles,
            "primary_vehicle": self.primary_vehicle,
            "history": self.history,
        }
        with open(USER_PROFILE_FILE, "w") as f:
            json.dump(data, f, indent=2)

    def add_vehicle(self, vehicle_name: str, braking_time_s: float):
        """Add or update a vehicle in the user's collection."""
        self.vehicles[vehicle_name] = {
            "braking_time": braking_time_s,
            "usage_count": self.vehicles.get(vehicle_name, {}).get("usage_count", 0),
        }
        # If this is the first vehicle, set it as primary
        if self.primary_vehicle is None:
            self.primary_vehicle = vehicle_name
        self._save()

    def remove_vehicle(self, vehicle_name: str):
        """Remove a vehicle from the collection."""
        if vehicle_name in self.vehicles:
            del self.vehicles[vehicle_name]
        if self.primary_vehicle == vehicle_name:
            self.primary_vehicle = self.get_most_used_vehicle()
        self._save()

    def record_usage(self, vehicle_name: str):
        """Record that a vehicle was used for analysis."""
        if vehicle_name not in self.vehicles:
            # Add with default settings if not already stored
            self.add_vehicle(vehicle_name, 3.5)
        self.vehicles[vehicle_name]["usage_count"] += 1
        self._save()

    def record_run(self, video_name: str, vehicle_name: str, score: float,
                   dist_score: float, lane_score: float, light_score: float,
                   duration_s: float, too_close: int, drifts: int, over_lines: int,
                   light_viols: int, stop_viols: int, collisions: int):
        """Record a safety analysis session into history."""
        run_record = {
            "id": f"{int(time.time())}_{len(self.history)}",
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "video_name": os.path.basename(video_name),
            "vehicle_name": vehicle_name,
            "overall_score": score,
            "distance_score": dist_score,
            "lane_score": lane_score,
            "light_score": light_score,
            "duration_s": duration_s,
            "infractions": {
                "too_close": too_close,
                "drifts": drifts,
                "over_lines": over_lines,
                "light_violations": light_viols,
                "stop_violations": stop_viols,
                "collision_warnings": collisions,
            }
        }
        self.history.append(run_record)
        # Keep only the last 50 runs to save space
        if len(self.history) > 50:
            self.history.pop(0)
        self._save()

    def get_history(self) -> List[dict]:
        """Return list of historical runs (newest first)."""
        return list(reversed(self.history))

    def clear_history(self):
        """Clear the run logs."""
        self.history = []
        self._save()

    def get_most_used_vehicle(self) -> Optional[str]:
        """Return the most frequently used vehicle."""
        if not self.vehicles:
            return None
        return max(self.vehicles.keys(), key=lambda v: self.vehicles[v].get("usage_count", 0))

    def get_vehicle_list(self) -> List[str]:
        """Return list of stored vehicles."""
        return sorted(list(self.vehicles.keys()))

    def get_suggested_vehicle(self) -> Optional[str]:
        """Return the suggested vehicle for next upload."""
        vehicle_list = self.get_vehicle_list()
        if len(vehicle_list) == 1:
            return vehicle_list[0]
        
        most_used = self.get_most_used_vehicle()
        if most_used and most_used in self.vehicles:
            return most_used
        
        return self.primary_vehicle

    def get_vehicle_braking_time(self, vehicle_name: str) -> float:
        """Get the recorded braking time for a vehicle."""
        if vehicle_name in self.vehicles:
            return self.vehicles[vehicle_name].get("braking_time", 3.5)
        return 3.5  # fallback default


In [ ]:
%%writefile analytics.py
"""
analytics.py  –  DriveSense AI  |  Telemetry Visualisation
===========================================================
Generates beautiful matplotlib charts from session events
for inclusion in the UI dashboard and reports.
"""

import matplotlib
matplotlib.use("Agg")  # thread-safe, non-interactive backend
import matplotlib.pyplot as plt
import os
import tempfile
from typing import Optional
from detector import FrameEvent


def generate_distance_chart(events: list[FrameEvent], safe_distance: float) -> Optional[str]:
    """
    Generate a telemetry line chart of Following Distance vs Safe Distance over time.
    Saves to a temporary file and returns its path.
    """
    valid_events = [e for e in events if e.nearest_car_dist_m is not None]
    if not valid_events:
        return None
        
    times = [e.timestamp_s for e in valid_events]
    distances = [e.nearest_car_dist_m for e in valid_events]
    
    # Matplotlib styling for dark glass theme
    plt.style.use('dark_background')
    fig, ax = plt.subplots(figsize=(10, 4), dpi=150)
    fig.patch.set_facecolor('#0a0a14')
    ax.set_facecolor('#10101e')
    
    # Plot distances
    ax.plot(times, distances, color='#3b5bdb', label='Following Distance', linewidth=2.5)
    
    # Plot safe distance line
    ax.axhline(y=safe_distance, color='#ef4444', linestyle='--', label='Safe Distance Buffer', linewidth=2)
    
    # Highlight unsafe regions (where following distance < safe distance)
    ax.fill_between(times, distances, safe_distance, where=[d < safe_distance for d in distances],
                    color='#ef4444', alpha=0.18, interpolate=True, label='Unsafe Zone')
    
    # Add title and labels
    ax.set_title("Following Distance Telemetry Profile", fontsize=13, fontweight='bold', pad=15, color='#e8e8ff')
    ax.set_xlabel("Time (seconds)", fontsize=10, labelpad=8, color='#aaa')
    ax.set_ylabel("Distance (metres)", fontsize=10, labelpad=8, color='#aaa')
    
    # Grid customization
    ax.grid(True, color='#1e1e35', linestyle=':', linewidth=0.8)
    
    # Legend
    legend = ax.legend(loc='upper right', frameon=True, facecolor='#10101e', edgecolor='#1e1e35')
    for text in legend.get_texts():
        text.set_color('#ccc')
        
    ax.tick_params(colors='#888', labelsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#1e1e35')
    ax.spines['bottom'].set_color('#1e1e35')
    
    # Save to temp file
    chart_path = os.path.join(tempfile.gettempdir(), f"drivesense_telemetry_chart.png")
    fig.tight_layout()
    fig.savefig(chart_path, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    return chart_path


In [ ]:
%%writefile app.py
"""
app.py  –  DriveSense AI  |  Main Web Dashboard
=============================================
Run:
    python app.py

A premium, high-tech, responsive Gradio Dashboard featuring:
  • Single Drive Safety Analysis with real-time gauges and advice
  • 1v1 Driving Battle Mode (compete against a friend or past runs)
  • Matplotlib Following Distance Telemetry Charting
  • Session History Logs & Comparators
  • Saved Vehicle Configurations
"""

import cv2
import gradio as gr
import numpy as np
import os
import subprocess
import tempfile
import time
from typing import Optional

from detector import DriveSenseDetector, DriverProfile, VEHICLE_FRONT_LENGTH_M, VEHICLE_DATABASE
from scorer import SafetyScorer, SafetyReport
from user_profile import UserProfile
from analytics import generate_distance_chart

# ─────────────────────────────────────────────
#  PIPELINE CONFIGURATION
# ─────────────────────────────────────────────
DESIRED_FPS        = 12
MODEL_NAME         = "yolo11n.pt"
GREEN_STATIONARY_S = 2.0

# Initialise persistent user profile
user_profile = UserProfile()


def _reencode_for_browser(raw_path: str) -> str:
    """
    Re-encode the OpenCV output video to H.264 using ffmpeg.
    Required for playback in HTML5 browser elements.
    """
    out_path = raw_path.replace(".mp4", "_h264.mp4")
    try:
        result = subprocess.run(
            [
                "ffmpeg", "-y",
                "-i", raw_path,
                "-vcodec", "libx264",
                "-pix_fmt", "yuv420p",
                "-preset", "fast",
                "-crf", "23",
                "-movflags", "+faststart",
                out_path,
            ],
            capture_output=True, timeout=300,
        )
        if result.returncode == 0 and os.path.exists(out_path):
            os.remove(raw_path)
            return out_path
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass
    return raw_path


def _run_pipeline(
    input_video    : str,
    vehicle_name   : str,
    braking_time_s : float,
    speed_limit    : float,
    processing_speed: str,
    progress,
    offset_pct     : float = 0.0,
    total_pct      : float = 1.0,
) -> tuple[str, SafetyReport, list]:
    """Runs the core CV processing pipeline over a video."""
    profile = DriverProfile(
        vehicle_name    = vehicle_name,
        braking_100_sec = braking_time_s,
        speed_limit_kmh = speed_limit,
    )

    detector = DriveSenseDetector(profile=profile, model_name=MODEL_NAME)
    scorer   = SafetyScorer(profile=profile)

    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise ValueError("Could not open video file.")

    # Map processing speed to target FPS
    fps_map = {
        "⚡ Fast (5 FPS)": 5.0,
        "⚖️ Standard (10 FPS)": 10.0,
        "🎯 High Precision (15 FPS)": 15.0
    }
    target_fps = fps_map.get(processing_speed, 10.0)

    orig_fps   = cap.get(cv2.CAP_PROP_FPS) or 25
    orig_fps   = max(1.0, orig_fps)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # Scale resolution down to max 1280px width to speed up CPU inference & resizing
    max_w = 1280
    scale_factor = 1.0
    if width > max_w:
        scale_factor = max_w / width
        width = max_w
        height = int(height * scale_factor)

    interval   = max(1, round(orig_fps / target_fps))
    output_fps = orig_fps / interval
    green_thresh = max(1, round(GREEN_STATIONARY_S * output_fps))

    out_dir = tempfile.gettempdir()
    out_name = f"drivesense_out_{int(time.time())}_{os.path.basename(input_video)}"
    out_path = os.path.join(out_dir, out_name)
    
    fourcc   = cv2.VideoWriter_fourcc(*"mp4v")
    out      = cv2.VideoWriter(out_path, fourcc, output_fps, (width, height))

    events   = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % interval == 0:
            if scale_factor < 1.0:
                frame = cv2.resize(frame, (width, height))
                
            event = detector.analyse_frame(
                frame, frame_idx,
                fps=orig_fps,
                green_frames_threshold=green_thresh,
            )
            events.append(event)

            if event.annotated_frame is not None:
                out.write(event.annotated_frame)
            else:
                out.write(frame)

            pct = offset_pct + (frame_idx / max(total_frames, 1)) * total_pct
            progress(min(0.99, pct), desc=f"Processing frames ({int(pct*100)}%)...")

        frame_idx += 1

    cap.release()
    out.release()
    
    out_path = _reencode_for_browser(out_path)
    report = scorer.score(events)
    
    return out_path, report, events


def process_video(
    input_video    : str,
    vehicle_name   : str,
    braking_time_s : float,
    speed_limit    : float,
    processing_speed: str,
    progress       = gr.Progress(track_tqdm=False),
):
    """Gradio entrypoint for analyzing a single drive."""
    if input_video is None:
        return None, "⚠️ Please upload a video file first.", "", None, gr.update(), ""

    try:
        out_path, report, events = _run_pipeline(
            input_video, vehicle_name, braking_time_s, speed_limit, processing_speed, progress, 0.0, 1.0
        )
    except Exception as e:
        return None, f"❌ Error processing video: {str(e)}", "", None, gr.update(), ""

    # Save details to profile database
    user_profile.record_usage(vehicle_name)
    user_profile.record_run(
        video_name   = input_video,
        vehicle_name = vehicle_name,
        score        = report.overall_score,
        dist_score   = report.distance_score,
        lane_score   = report.lane_score,
        light_score  = report.light_score,
        duration_s   = report.duration_s,
        too_close    = report.too_close_count,
        drifts       = report.drift_events,
        over_lines   = report.over_line_events,
        light_viols  = report.light_violations,
        stop_viols   = report.stop_sign_violations,
        collisions   = report.collision_warnings,
    )

    # Generate telemetry chart
    v = speed_limit / 3.6
    t_reaction = 1.5
    v100 = 100 / 3.6
    a = v100 / max(braking_time_s, 0.5)
    safe_d = (v * t_reaction + (v ** 2) / (2 * a)) * 1.2
    chart_path = generate_distance_chart(events, safe_d)

    score_html = _build_score_html(report)
    history_choices, history_table = load_history_ui()

    return (
        out_path, 
        report.markdown, 
        score_html, 
        chart_path, 
        gr.update(choices=history_choices), 
        history_table
    )


def process_1v1_battle(
    video_1: str, name_1: str, vehicle_1: str, braking_1: float,
    video_2: str, name_2: str, vehicle_2: str, braking_2: float,
    speed_limit: float,
    processing_speed: str,
    progress = gr.Progress(track_tqdm=False)
):
    """Gradio entrypoint for 1v1 battle mode."""
    if not video_1 or not video_2:
        return None, None, "⚠️ Please upload both videos to start the battle.", "", gr.update(), ""
        
    name_1 = name_1 or "Driver 1"
    name_2 = name_2 or "Driver 2"

    try:
        # Driver 1 (0% to 50%)
        out_1, report_1, _ = _run_pipeline(
            video_1, vehicle_1, braking_1, speed_limit, processing_speed, progress, 0.0, 0.5
        )
        user_profile.record_usage(vehicle_1)
        user_profile.record_run(
            video_name   = video_1,
            vehicle_name = f"{name_1} ({vehicle_1})",
            score        = report_1.overall_score,
            dist_score   = report_1.distance_score,
            lane_score   = report_1.lane_score,
            light_score  = report_1.light_score,
            duration_s   = report_1.duration_s,
            too_close    = report_1.too_close_count,
            drifts       = report_1.drift_events,
            over_lines   = report_1.over_line_events,
            light_viols  = report_1.light_violations,
            stop_viols   = report_1.stop_sign_violations,
            collisions   = report_1.collision_warnings,
        )

        # Driver 2 (50% to 100%)
        out_2, report_2, _ = _run_pipeline(
            video_2, vehicle_2, braking_2, speed_limit, processing_speed, progress, 0.5, 0.5
        )
        user_profile.record_usage(vehicle_2)
        user_profile.record_run(
            video_name   = video_2,
            vehicle_name = f"{name_2} ({vehicle_2})",
            score        = report_2.overall_score,
            dist_score   = report_2.distance_score,
            lane_score   = report_2.lane_score,
            light_score  = report_2.light_score,
            duration_s   = report_2.duration_s,
            too_close    = report_2.too_close_count,
            drifts       = report_2.drift_events,
            over_lines   = report_2.over_line_events,
            light_viols  = report_2.light_violations,
            stop_viols   = report_2.stop_sign_violations,
            collisions   = report_2.collision_warnings,
        )
        
    except Exception as e:
        return None, None, f"❌ Error processing battle: {str(e)}", "", gr.update(), ""

    battle_html = _build_battle_html(name_1, report_1, name_2, report_2)
    battle_md = _build_battle_markdown(name_1, report_1, name_2, report_2)
    
    choices, history_table = load_history_ui()
    
    return out_1, out_2, battle_html, battle_md, gr.update(choices=choices), history_table


# ─────────────────────────────────────────────
#  HTML & MARKDOWN GENERATION
# ─────────────────────────────────────────────
def _score_color(s: float) -> str:
    if s >= 8:  return "#22c55e" # Neon Green
    if s >= 6:  return "#eab308" # Neon Yellow
    return "#ef4444"             # Neon Red


def _build_score_html(report: SafetyReport) -> str:
    def gauge(label, score):
        pct   = score * 10
        color = _score_color(score)
        return f"""
<div style="margin:12px 0;">
  <div style="display:flex;justify-content:space-between;margin-bottom:4px;">
    <span style="font-family:'Courier New',monospace;font-size:12px;color:#aaa;">{label}</span>
    <span style="font-family:'Courier New',monospace;font-size:12px;color:{color};font-weight:bold;">{score:.1f}/10</span>
  </div>
  <div style="background:#1e1e2f;border-radius:6px;height:10px;overflow:hidden;border:1px solid #26263e;">
    <div style="width:{pct}%;height:100%;background:{color};border-radius:6px;
                box-shadow: 0 0 8px {color}88; transition:width 0.8s ease;"></div>
  </div>
</div>"""

    overall_color = _score_color(report.overall_score)
    grade_map = [(9,"A+"),(8,"A"),(7,"B"),(6,"C"),(5,"D"),(0,"F")]
    grade = next(g for t,g in grade_map if report.overall_score >= t)

    html = f"""
<div style="background:#0f0f1c;border:1px solid #2a2a44;border-radius:12px;
            padding:24px 28px;font-family:'Courier New',monospace;color:#e0e0f5;max-width:100%;">

  <div style="text-align:center;margin-bottom:20px;">
    <div style="font-size:11px;letter-spacing:4px;color:#888;margin-bottom:6px;">SAFETY SCORE</div>
    <div style="font-size:60px;font-weight:900;color:{overall_color};line-height:1;
                text-shadow: 0 0 15px {overall_color}55;">{report.overall_score}</div>
    <div style="font-size:18px;color:{overall_color};margin-top:4px;">/ 10 &nbsp;({grade})</div>
    <div style="font-size:11px;color:#777;margin-top:8px;">
      {report.duration_s:.1f}s  ·  {report.analysed_frames} frames  ·  {report.vehicle_name}
    </div>
  </div>

  <div style="border-top:1px solid #26263e;padding-top:16px;">
    {gauge("🚘  Following Gap & TTC", report.distance_score)}
    {gauge("🛣️  Lane Keeping Stability", report.lane_score)}
    {gauge("🚦  Intersection Response", report.light_score)}
  </div>

  <div style="border-top:1px solid #26263e;padding-top:14px;margin-top:10px;
              font-size:11px;color:#888;display:grid;grid-template-columns:1fr 1fr;gap:8px 16px;">
    <div>Too-close frames: <b style="color:#ccc;">{report.too_close_count}</b></div>
    <div>Collision alerts: <b style="color:#ccc;">{report.collision_warnings}</b></div>
    <div>Lane drifts: <b style="color:#ccc;">{report.drift_events}</b></div>
    <div>Over-line events: <b style="color:#ccc;">{report.over_line_events}</b></div>
    <div>Signal delays: <b style="color:#ccc;">{report.light_violations}</b></div>
    <div>Stop sign rolls: <b style="color:#ccc;">{report.stop_sign_violations}</b></div>
  </div>

</div>"""
    return html


def _build_battle_html(name_1: str, report_1, name_2: str, report_2) -> str:
    s1 = report_1.overall_score
    s2 = report_2.overall_score
    
    if s1 > s2:
        winner_text = f"🏆 {name_1} wins! ({s1:.1f} vs {s2:.1f})"
        w_color = "#22c55e"
    elif s2 > s1:
        winner_text = f"🏆 {name_2} wins! ({s2:.1f} vs {s1:.1f})"
        w_color = "#22c55e"
    else:
        winner_text = f"👔 It's a Tie! ({s1:.1f} vs {s2:.1f})"
        w_color = "#eab308"
        
    def score_badge(score):
        color = _score_color(score)
        return f'<span style="color:{color};font-weight:bold;font-size:24px;text-shadow:0 0 8px {color}44;">{score:.1f}/10</span>'
        
    html = f"""
<div style="background:#0f0f1c;border:1px solid #2a2a44;border-radius:12px;
            padding:24px;font-family:'Courier New',monospace;color:#e0e0f5;max-width:100%;">

  <div style="text-align:center;margin-bottom:24px;padding:14px;background:#151528;border-radius:8px;border:1px solid #303054;">
    <div style="font-size:11px;letter-spacing:4px;color:#888;margin-bottom:6px;">BATTLE OUTCOME</div>
    <div style="font-size:32px;font-weight:900;color:{w_color};text-shadow:0 0 15px {w_color}55;">{winner_text}</div>
  </div>

  <div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;margin-bottom:20px;">
    <!-- Driver 1 -->
    <div style="background:#151528;border:1px solid #26263e;border-radius:8px;padding:16px;text-align:center;">
      <div style="font-size:20px;font-weight:bold;color:#4c6ef5;margin-bottom:4px;">{name_1}</div>
      <div style="font-size:11px;color:#888;margin-bottom:12px;">{report_1.vehicle_name}</div>
      <div>{score_badge(s1)}</div>
      <div style="margin-top:14px;text-align:left;font-size:11px;line-height:1.7;">
        <div style="display:flex;justify-content:space-between;border-bottom:1px solid #222;padding:2px 0;"><span>Gap Score:</span><b style="color:#ccc;">{report_1.distance_score:.1f}</b></div>
        <div style="display:flex;justify-content:space-between;border-bottom:1px solid #222;padding:2px 0;"><span>Lane Score:</span><b style="color:#ccc;">{report_1.lane_score:.1f}</b></div>
        <div style="display:flex;justify-content:space-between;padding:2px 0;"><span>Intersection:</span><b style="color:#ccc;">{report_1.light_score:.1f}</b></div>
      </div>
    </div>
    
    <!-- Driver 2 -->
    <div style="background:#151528;border:1px solid #26263e;border-radius:8px;padding:16px;text-align:center;">
      <div style="font-size:20px;font-weight:bold;color:#7048e8;margin-bottom:4px;">{name_2}</div>
      <div style="font-size:11px;color:#888;margin-bottom:12px;">{report_2.vehicle_name}</div>
      <div>{score_badge(s2)}</div>
      <div style="margin-top:14px;text-align:left;font-size:11px;line-height:1.7;">
        <div style="display:flex;justify-content:space-between;border-bottom:1px solid #222;padding:2px 0;"><span>Gap Score:</span><b style="color:#ccc;">{report_2.distance_score:.1f}</b></div>
        <div style="display:flex;justify-content:space-between;border-bottom:1px solid #222;padding:2px 0;"><span>Lane Score:</span><b style="color:#ccc;">{report_2.lane_score:.1f}</b></div>
        <div style="display:flex;justify-content:space-between;padding:2px 0;"><span>Intersection:</span><b style="color:#ccc;">{report_2.light_score:.1f}</b></div>
      </div>
    </div>
  </div>

  <div style="border-top:1px solid #26263e;padding-top:16px;">
    <div style="font-size:14px;font-weight:bold;color:#aaa;margin-bottom:12px;text-align:center;letter-spacing:2px;">DRIVING ENVELOPE METRICS</div>
    
    <table style="width:100%;font-size:12px;border-collapse:collapse;text-align:left;">
      <thead>
        <tr style="border-bottom:1px solid #26263e;color:#888;height:30px;">
          <th style="padding:6px 0;">Driving Metric</th>
          <th style="padding:6px 0;text-align:center;color:#4c6ef5;width:30%;">{name_1}</th>
          <th style="padding:6px 0;text-align:center;color:#7048e8;width:30%;">{name_2}</th>
        </tr>
      </thead>
      <tbody>
        <tr style="border-bottom:1px solid #1a1a2e;height:32px;">
          <td style="padding:6px 0;color:#aaa;">Tailgate Frames</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_1.too_close_count > report_2.too_close_count else '#aaa'}">{report_1.too_close_count}</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_2.too_close_count > report_1.too_close_count else '#aaa'}">{report_2.too_close_count}</td>
        </tr>
        <tr style="border-bottom:1px solid #1a1a2e;height:32px;">
          <td style="padding:6px 0;color:#aaa;">Collision Alerts</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_1.collision_warnings > report_2.collision_warnings else '#aaa'}">{report_1.collision_warnings}</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_2.collision_warnings > report_1.collision_warnings else '#aaa'}">{report_2.collision_warnings}</td>
        </tr>
        <tr style="border-bottom:1px solid #1a1a2e;height:32px;">
          <td style="padding:6px 0;color:#aaa;">Lane Drifts</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_1.drift_events > report_2.drift_events else '#aaa'}">{report_1.drift_events}</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_2.drift_events > report_1.drift_events else '#aaa'}">{report_2.drift_events}</td>
        </tr>
        <tr style="border-bottom:1px solid #1a1a2e;height:32px;">
          <td style="padding:6px 0;color:#aaa;">Lane Straddles</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_1.over_line_events > report_2.over_line_events else '#aaa'}">{report_1.over_line_events}</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_2.over_line_events > report_1.over_line_events else '#aaa'}">{report_2.over_line_events}</td>
        </tr>
        <tr style="border-bottom:1px solid #1a1a2e;height:32px;">
          <td style="padding:6px 0;color:#aaa;">Signal Violations</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_1.light_violations > report_2.light_violations else '#aaa'}">{report_1.light_violations}</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_2.light_violations > report_1.light_violations else '#aaa'}">{report_2.light_violations}</td>
        </tr>
        <tr style="border-bottom:1px solid #26263e;height:32px;">
          <td style="padding:6px 0;color:#aaa;">Stop Sign Rolls</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_1.stop_sign_violations > report_2.stop_sign_violations else '#aaa'}">{report_1.stop_sign_violations}</td>
          <td style="padding:6px 0;text-align:center;color:{'#ef4444' if report_2.stop_sign_violations > report_1.stop_sign_violations else '#aaa'}">{report_2.stop_sign_violations}</td>
        </tr>
      </tbody>
    </table>
  </div>
</div>
"""
    return html


def _build_battle_markdown(name_1: str, report_1, name_2: str, report_2) -> str:
    s1 = report_1.overall_score
    s2 = report_2.overall_score
    
    if s1 > s2:
        winner = name_1
        loser = name_2
        diff = s1 - s2
        summary = f"🏆 **{winner}** outperformed **{loser}** by **{diff:.1f}** safety points, demonstrating superior road discipline."
    elif s2 > s1:
        winner = name_2
        loser = name_1
        diff = s2 - s1
        summary = f"🏆 **{winner}** outperformed **{loser}** by **{diff:.1f}** safety points, demonstrating superior road discipline."
    else:
        summary = "👔 Both drivers showed identical safety ratings in this match. Excellent job maintaining a steady safety envelope!"

    lines = [
        f"# ⚔️ 1v1 Battle Scoreboard: {name_1} vs {name_2}",
        f"",
        summary,
        f"",
        f"### Summary of infractions:",
        f"- **{name_1}** driving **{report_1.vehicle_name}** logged:",
        f"  - Tailgating events: **{report_1.too_close_count}** frames",
        f"  - Forward Collision warnings: **{report_1.collision_warnings}** events",
        f"  - Lane discipline issues: **{report_1.drift_events + report_1.over_line_events}** times",
        f"  - Intersection infractions: **{report_1.light_violations + report_1.stop_sign_violations}** times",
        f"",
        f"- **{name_2}** driving **{report_2.vehicle_name}** logged:",
        f"  - Tailgating events: **{report_2.too_close_count}** frames",
        f"  - Forward Collision warnings: **{report_2.collision_warnings}** events",
        f"  - Lane discipline issues: **{report_2.drift_events + report_2.over_line_events}** times",
        f"  - Intersection infractions: **{report_2.light_violations + report_2.stop_sign_violations}** times",
        f"",
        f"---",
        f"*Generated by DriveSense AI 1v1 Arena.*"
    ]
    return "\n".join(lines)


# ─────────────────────────────────────────────
#  HISTORY AND PRESETS
# ─────────────────────────────────────────────
def load_history_ui() -> tuple[list, str]:
    """Load history and format it into a table and dropdown choices."""
    history = user_profile.get_history()
    if not history:
        return [], "<div style='text-align:center;padding:30px;color:#555;font-family:monospace;'>No driving records logged. Analyze a video to get started!</div>"
        
    html = """
<div style="font-family:'Courier New',monospace;color:#e0e0f5;max-width:100%;">
  <table style="width:100%;font-size:12px;border-collapse:collapse;text-align:left;">
    <thead>
      <tr style="border-bottom:1px solid #26263e;color:#888;height:32px;">
        <th style="padding:8px 0;">Timestamp</th>
        <th style="padding:8px 0;">Session Title (Video Source)</th>
        <th style="padding:8px 0;">Vehicle used</th>
        <th style="padding:8px 0;text-align:center;">Safety Rating</th>
      </tr>
    </thead>
    <tbody>
    """
    choices = []
    for idx, run in enumerate(history):
        score_color = _score_color(run["overall_score"])
        html += f"""
      <tr style="border-bottom:1px solid #141424;height:36px;transition: background 0.2s;">
        <td style="color:#777;padding:8px 0;">{run["timestamp"]}</td>
        <td style="color:#ccc;font-weight:bold;padding:8px 0;">{run["video_name"]}</td>
        <td style="color:#aaa;padding:8px 0;">{run["vehicle_name"]}</td>
        <td style="text-align:center;color:{score_color};font-weight:bold;font-size:13px;padding:8px 0;">{run["overall_score"]:.1f}/10</td>
      </tr>
        """
        label = f"{run['timestamp']} - {run['video_name']} ({run['overall_score']:.1f}/10)"
        choices.append((label, run["id"]))
        
    html += """
    </tbody>
  </table>
</div>
    """
    return choices, html


def build_vehicle_presets():
    """Dynamically build presets list from database & user saved vehicles."""
    presets = {}
    
    # Stored vehicles
    user_vehicles = user_profile.get_vehicle_list()
    if user_vehicles:
        for vname in user_vehicles:
            usage = user_profile.vehicles[vname].get("usage_count", 0)
            braking = user_profile.vehicles[vname].get("braking_time", 3.5)
            label = f"📌 {vname} (used {usage} times)"
            presets[label] = (vname, braking)
        presets["─" * 40] = ("Standard Car", 3.5)
    
    # Default database
    for vname in sorted(VEHICLE_DATABASE.keys()):
        if vname not in user_profile.vehicles:
            presets[vname] = (vname, 3.5)
    
    presets["🔧 Custom (enter below)"] = ("Custom Vehicle", 3.5)
    return presets

VEHICLE_PRESETS = build_vehicle_presets()


def update_braking_from_preset(preset_label):
    vname, braking = VEHICLE_PRESETS.get(preset_label, ("Custom Vehicle", 3.5))
    if vname in user_profile.vehicles:
        braking = user_profile.vehicles[vname].get("braking_time", braking)
    return braking, vname


def get_suggested_vehicle() -> str:
    suggested = user_profile.get_suggested_vehicle()
    if suggested:
        for label, (vname, _) in VEHICLE_PRESETS.items():
            if vname == suggested:
                return label
    return next(iter(VEHICLE_PRESETS.keys()))


# ─────────────────────────────────────────────
#  TAB ACTIONS AND HANDLERS
# ─────────────────────────────────────────────
def add_new_vehicle_action(name, braking):
    if not name or name.strip() == "":
        return gr.update(), "⚠️ Vehicle name cannot be blank."
    user_profile.add_vehicle(name, float(braking))
    
    # Rebuild preset choices in app
    global VEHICLE_PRESETS
    VEHICLE_PRESETS = build_vehicle_presets()
    new_choices = list(VEHICLE_PRESETS.keys())
    
    return gr.update(choices=new_choices, value=get_suggested_vehicle()), f"✅ Saved {name} to profile database."


def clear_history_action():
    user_profile.clear_history()
    choices, table = load_history_ui()
    return gr.update(choices=choices), table


# ─────────────────────────────────────────────
#  GRADIO CUSTOM STYLING (CSS)
# ─────────────────────────────────────────────
CSS = """
/* Dark cyberpunk glow theme */
body, .gradio-container {
    background-color: #05050d !important;
    background-image: radial-gradient(circle at 50% 50%, #120e26 0%, #05050d 100%) !important;
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", sans-serif !important;
    color: #e2e2f0 !important;
}

.gradio-container {
    max-width: 1200px !important;
    margin: 0 auto;
}

.panel-card {
    background: rgba(15, 15, 28, 0.72) !important;
    border: 1px solid #252542 !important;
    border-radius: 12px !important;
    padding: 22px !important;
    box-shadow: 0 10px 30px rgba(0, 0, 0, 0.5) !important;
    backdrop-filter: blur(10px);
}

.ds-header {
    text-align: center;
    padding: 30px 0 20px;
}
.ds-header h1 {
    font-family: 'Courier New', monospace;
    font-size: 2.5rem;
    font-weight: 900;
    color: #f3f3ff;
    text-shadow: 0 0 15px rgba(76, 110, 245, 0.5);
    margin: 0;
    letter-spacing: 0.15em;
}
.ds-header p {
    font-family: 'Courier New', monospace;
    color: #6a6a9d;
    font-size: 0.85rem;
    margin: 8px 0 0;
    letter-spacing: 0.25em;
    text-transform: uppercase;
}

button.primary-btn, .gr-button-primary {
    background: linear-gradient(135deg, #4c6ef5 0%, #7048e8 100%) !important;
    border: none !important;
    font-family: 'Courier New', monospace !important;
    letter-spacing: 0.15em !important;
    font-weight: 700 !important;
    border-radius: 8px !important;
    color: #ffffff !important;
    box-shadow: 0 4px 15px rgba(112, 72, 232, 0.4) !important;
    transition: all 0.3s ease !important;
}
button.primary-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 6px 20px rgba(112, 72, 232, 0.6) !important;
    opacity: 0.95 !important;
}

label span {
    font-family: 'Courier New', monospace !important;
    color: #8f8faf !important;
    font-size: 11px !important;
    letter-spacing: 0.08em !important;
}

.gr-tabs {
    border-bottom: 2px solid #22223a !important;
}
.tab-nav button {
    font-family: 'Courier New', monospace !important;
    font-size: 13px !important;
    letter-spacing: 0.08em !important;
    color: #6a6a9d !important;
    background: transparent !important;
    border: none !important;
}
.tab-nav button.selected {
    color: #e2e2f0 !important;
    border-bottom: 2px solid #4c6ef5 !important;
    font-weight: bold !important;
}
"""

# ─────────────────────────────────────────────
#  GRADIO APPLICATION LAYOUT
# ─────────────────────────────────────────────
with gr.Blocks(css=CSS, title="DriveSense AI Dashboard") as demo:

    # ── Header ──────────────────────────────────
    gr.HTML("""
    <div class="ds-header">
      <h1>⬡ DRIVESENSE AI</h1>
      <p>ADAS Driver Evaluation · Telemetry Charting · 1v1 Battle Arena</p>
    </div>
    """)

    # Setup starting UI choices
    initial_history_choices, initial_history_table = load_history_ui()

    with gr.Tabs():
        
        # ── TAB 1: SINGLE DRIVE ANALYZER ────────────
        with gr.Tab("🚗 Single Drive Evaluation"):
            with gr.Row():
                
                # Left inputs
                with gr.Column(scale=1):
                    gr.HTML('<div class="panel-card">')
                    gr.Markdown("### 📹 Upload Dashcam / Bumper Video")
                    video_input = gr.Video(
                        label="Source Footage (supports mp4/avi)",
                        height=210,
                    )
                    gr.HTML('</div><br>')

                    gr.HTML('<div class="panel-card">')
                    gr.Markdown("### ⚙️ Calibration Settings")

                    vehicle_preset = gr.Dropdown(
                        choices=list(VEHICLE_PRESETS.keys()),
                        value=get_suggested_vehicle(),
                        label="🚗 Vehicle Config Database",
                    )
                    vehicle_name_box = gr.Textbox(
                        value=VEHICLE_PRESETS[get_suggested_vehicle()][0],
                        label="Report Vehicle Identifier (Auto-filled)",
                        max_lines=1,
                    )
                    braking_slider = gr.Slider(
                        minimum=2.0, maximum=8.0, value=VEHICLE_PRESETS[get_suggested_vehicle()][1], step=0.1,
                        label="0–100 km/h Braking Profile (seconds)",
                    )
                    speed_limit_slider = gr.Slider(
                        minimum=30, maximum=130, value=70, step=5,
                        label="Road Speed Limit (km/h)",
                    )
                    processing_speed_dropdown = gr.Dropdown(
                        choices=["⚡ Fast (5 FPS)", "⚖️ Standard (10 FPS)", "🎯 High Precision (15 FPS)"],
                        value="⚖️ Standard (10 FPS)",
                        label="Processing Speed & Quality",
                    )
                    gr.HTML('</div><br>')

                    analyse_btn = gr.Button(
                        "▶  START EVALUATION",
                        variant="primary",
                        elem_classes=["primary-btn"],
                    )

                # Right outputs
                with gr.Column(scale=2):
                    with gr.Row():
                        with gr.Column(scale=1):
                            gr.HTML('<div class="panel-card">')
                            gr.Markdown("### 🎬 Annotated ADAS Stream")
                            video_output = gr.Video(label="ADAS Stream Overlay", height=320)
                            gr.HTML('</div>')
                            
                        with gr.Column(scale=1):
                            gr.HTML('<div class="panel-card">')
                            gr.Markdown("### 📊 Active Safety Scorecard")
                            score_html = gr.HTML(value="<div style='text-align:center;padding:60px;color:#444;font-family:monospace;'>Upload a video and start evaluation to display safety scorecard.</div>")
                            gr.HTML('</div>')
                    
                    gr.HTML('<br><div class="panel-card">')
                    gr.Markdown("### 📈 Following Gap Telemetry Profile")
                    telemetry_chart = gr.Image(label="Telemetry Plot", show_label=False)
                    gr.HTML('</div><br>')
                    
                    gr.HTML('<div class="panel-card">')
                    gr.Markdown("### 📋 Evaluator Safety Feedback")
                    report_md = gr.Markdown(value="*Evaluation report will be printed here.*")
                    gr.HTML('</div>')

        # ── TAB 2: 1v1 BATTLE ARENA ─────────────────
        with gr.Tab("⚔️ 1v1 Battle Arena"):
            gr.Markdown("### Compare driving parameters and declare a winner in a head-to-head match!")
            
            with gr.Row():
                # Driver 1 config
                with gr.Column(scale=1):
                    gr.HTML('<div class="panel-card" style="border: 1px solid #3b5bdb55 !important;">')
                    gr.Markdown("### 🚘 Driver A (Challenger 1)")
                    b_driver_name_1 = gr.Textbox(value="Challenger A", label="Driver Name")
                    b_video_1 = gr.Video(label="Driver A Clip", height=180)
                    b_preset_1 = gr.Dropdown(
                        choices=list(VEHICLE_PRESETS.keys()),
                        value=get_suggested_vehicle(),
                        label="Driver A Vehicle",
                    )
                    b_braking_1 = gr.Slider(minimum=2.0, maximum=8.0, value=3.5, step=0.1, label="Driver A Braking (s)")
                    gr.HTML('</div>')
                
                # Driver 2 config
                with gr.Column(scale=1):
                    gr.HTML('<div class="panel-card" style="border: 1px solid #7048e855 !important;">')
                    gr.Markdown("### 🚘 Driver B (Challenger 2)")
                    b_driver_name_2 = gr.Textbox(value="Challenger B", label="Driver Name")
                    b_video_2 = gr.Video(label="Driver B Clip", height=180)
                    b_preset_2 = gr.Dropdown(
                        choices=list(VEHICLE_PRESETS.keys()),
                        value=get_suggested_vehicle(),
                        label="Driver B Vehicle",
                    )
                    b_braking_2 = gr.Slider(minimum=2.0, maximum=8.0, value=3.5, step=0.1, label="Driver B Braking (s)")
                    gr.HTML('</div>')
            
            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML('<br><div class="panel-card">')
                    gr.Markdown("### ⚙️ Shared Environmental Settings")
                    b_speed_limit = gr.Slider(minimum=30, maximum=130, value=70, step=5, label="Shared Road Speed Limit (km/h)")
                    b_processing_speed = gr.Dropdown(
                        choices=["⚡ Fast (5 FPS)", "⚖️ Standard (10 FPS)", "🎯 High Precision (15 FPS)"],
                        value="⚖️ Standard (10 FPS)",
                        label="Shared Processing Speed & Quality",
                    )
                    gr.HTML('</div><br>')
                    
                    battle_btn = gr.Button(
                        "⚔️ START 1v1 BATTLE",
                        variant="primary",
                        elem_classes=["primary-btn"],
                    )
                
                with gr.Column(scale=2):
                    pass
            
            # Battle Results
            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML('<br><div class="panel-card">')
                    gr.Markdown("### 🎬 Challenger A Output Stream")
                    b_video_out_1 = gr.Video(label="Driver A Annotated", height=240)
                    gr.HTML('</div>')
                with gr.Column(scale=1):
                    gr.HTML('<br><div class="panel-card">')
                    gr.Markdown("### 🎬 Challenger B Output Stream")
                    b_video_out_2 = gr.Video(label="Driver B Annotated", height=240)
                    gr.HTML('</div>')
            
            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML('<br><div class="panel-card">')
                    gr.Markdown("### 📊 Head-to-Head Comparison Card")
                    battle_html_out = gr.HTML(value="<div style='text-align:center;padding:40px;color:#444;font-family:monospace;'>Initiate a match to see the battle outcome.</div>")
                    gr.HTML('</div>')
                with gr.Column(scale=1):
                    gr.HTML('<br><div class="panel-card">')
                    gr.Markdown("### 📝 Combat Log Details")
                    battle_md_out = gr.Markdown(value="*Results will print here.*")
                    gr.HTML('</div>')

        # ── TAB 3: HISTORY & ANALYTICS ──────────────
        with gr.Tab("📊 Analytics & History Log"):
            with gr.Row():
                with gr.Column(scale=3):
                    gr.HTML('<div class="panel-card">')
                    gr.Markdown("### 📋 Historical Evaluation Records")
                    history_table_out = gr.HTML(value=initial_history_table)
                    gr.HTML('</div>')
                
                with gr.Column(scale=1):
                    gr.HTML('<div class="panel-card">')
                    gr.Markdown("### 🛠️ History Actions")
                    clear_hist_btn = gr.Button("🗑️ CLEAR RUN LOGS")
                    gr.HTML('</div>')

        # ── TAB 4: VEHICLE PROFILES ─────────────────
        with gr.Tab("🔧 Vehicle Manager"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML('<div class="panel-card">')
                    gr.Markdown("### ➕ Add Custom Vehicle Profile")
                    new_vehicle_name = gr.Textbox(placeholder="E.g., Honda Civic 2024", label="Vehicle Display Name")
                    new_braking_slider = gr.Slider(minimum=2.0, maximum=8.0, value=3.5, step=0.1, label="0-100 km/h braking time (s)")
                    save_vehicle_btn = gr.Button("💾 SAVE VEHICLE", variant="primary")
                    gr.HTML('</div>')
                
                with gr.Column(scale=2):
                    gr.HTML('<div class="panel-card">')
                    gr.Markdown("### 📖 Active Calibration Specifications")
                    gr.HTML("""
                    <div style="font-family:'Courier New',monospace;font-size:12px;color:#aaa;line-height:1.8;">
                      <b>Distance Calibration Model:</b> Pinhole Camera Model via Triangle Similarity<br>
                      <b>Physics Engine Constants:</b><br>
                      - Focal Length Constant: 800 px·m/px (Calibrated for 1080p lens at ~60° HFOV)<br>
                      - Reaction Leniency Multiplier: 1.2x (Pakistani traffic norms correction factor)<br>
                      - Human Reaction Speed: 1.5 seconds default safe reaction buffer<br><br>
                      <b>Representative Object Width Standards:</b><br>
                      - Pedestrians: 0.55m  |  Motorcycles: 0.80m  |  Bicycles: 0.65m<br>
                      - Standard Hatchbacks/Sedans: 1.60m - 1.80m<br>
                      - Buses and Heavy Trucks: 2.50m
                    </div>
                    """)
                    gr.HTML('</div>')

    # ── Wire preset dropdowns to update sliders ──
    vehicle_preset.change(
        fn=update_braking_from_preset,
        inputs=[vehicle_preset],
        outputs=[braking_slider, vehicle_name_box],
    )
    
    b_preset_1.change(
        fn=update_braking_from_preset,
        inputs=[b_preset_1],
        outputs=[b_braking_1, gr.State()], # Throwaway name
    )
    
    b_preset_2.change(
        fn=update_braking_from_preset,
        inputs=[b_preset_2],
        outputs=[b_braking_2, gr.State()],
    )

    # ── Wire Evaluate Button ─────────────────────
    # Outputs: video_output, report_md, score_html, telemetry_chart, vehicle_preset choice, history_table_out
    analyse_btn.click(
        fn=process_video,
        inputs=[
            video_input,
            vehicle_name_box,
            braking_slider,
            speed_limit_slider,
            processing_speed_dropdown,
        ],
        outputs=[
            video_output,
            report_md,
            score_html,
            telemetry_chart,
            vehicle_preset, # Update dropdown choices in both tabs
            history_table_out,
        ],
    )

    # ── Wire 1v1 Battle Button ───────────────────
    battle_btn.click(
        fn=process_1v1_battle,
        inputs=[
            b_video_1, b_driver_name_1, b_preset_1, b_braking_1,
            b_video_2, b_driver_name_2, b_preset_2, b_braking_2,
            b_speed_limit,
            b_processing_speed,
        ],
        outputs=[
            b_video_out_1,
            b_video_out_2,
            battle_html_out,
            battle_md_out,
            vehicle_preset,
            history_table_out
        ],
    )

    # ── Wire Vehicle Profiles Manager ────────────
    save_vehicle_btn.click(
        fn=add_new_vehicle_action,
        inputs=[new_vehicle_name, new_braking_slider],
        outputs=[vehicle_preset, report_md], # Shows success message in feedback box
    )
    
    # Reload preset lists when save button clicked
    def refresh_dropdowns():
        presets = build_vehicle_presets()
        return gr.update(choices=list(presets.keys())), gr.update(choices=list(presets.keys()))
        
    save_vehicle_btn.click(
        fn=refresh_dropdowns,
        outputs=[b_preset_1, b_preset_2]
    )

    # ── Wire Clear History Actions ───────────────
    clear_hist_btn.click(
        fn=clear_history_action,
        outputs=[vehicle_preset, history_table_out], # Updates history dropdown and history table
    )

    # ── Footer ────────────────────────────────────
    gr.HTML("""
    <div style="text-align:center;padding:30px 0 10px;
         font-family:'Courier New',monospace;font-size:11px;
         color:#4a4a6d;letter-spacing:0.12em;">
      DRIVESENSE AI · SOFTWARE ENGINEERING SEMESTER SUBMISSION · ITU LAHORE BSAI
    </div>
    """)


#  ENTRY POINT
if __name__ == "__main__":
    demo.launch(
        share=True,
        server_name="0.0.0.0",
        server_port=7860,
        show_error=True,
    )

In [ ]:
# ── Cell 8: Verify all files exist ────────────────────────────────────────────
import os
for f in ['app.py', 'detector.py', 'scorer.py', 'user_profile.py', 'analytics.py']:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f'  {f}: {size} bytes ✅')
    else:
        print(f'  {f}: ❌ MISSING!')
print('\nAll files ready.')

In [ ]:
# ── Cell 9: LAUNCH THE APP ────────────────────────────────────────────────────
# This cell starts the server. A public Gradio URL will appear below.
# Click the .gradio.live link to open DriveSense AI in a new tab.
!python app.py